# Baseline 1: Mean-difference probe (white box)

This notebook implements the mean-difference probe from Goldowsky-Dill et al. [Detecting Strategic Deception Using Linear Probes](https://arxiv.org/abs/2502.03407).

The method takes the residual-stream activation of a model's response, averages it separately over deceptive and honest examples, and subtracts. That difference vector is the probe. It scores a new response by projecting its activation onto the direction $s(\mathbf{h}) = \mathbf{h} \cdot \mathbf{d}$.

Following the reference implementation ([`deception-detection`](https://github.com/ApolloResearch/deception-detection), `MMSDetector`), we **train** each probe on the RepE honest/deceptive *true-facts* contrast set (`repe_honesty`) and **evaluate by transfer** on Liars' Bench.

We train **one probe per model** — `google/gemma-3-27b-it`, `Qwen/Qwen3.5-27B`, and `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16` — using each model's own activations, evaluate **on-policy** (each model's own Liars' Bench generations), and **save the three probes separately**. The RepE training *text* is shared; activations, layer selection, and weights are per-model.

## 0️⃣ Setup

In [1]:
# nnsight — hackathon build (PEFT/LoRA + remote NDIF support)
!pip install -q git+https://github.com/ndif-team/nnsight.git@hackathon/peft
!pip install -q peft
!pip install -q --upgrade torchao "transformers==5.9.0"

from IPython.display import clear_output
clear_output()

In [ ]:
import os
from nnsight import CONFIG
from huggingface_hub import login

NDIF_API_KEY = os.environ.get("NDIF_API_KEY", "YOUR_NDIF_API_KEY")
HF_TOKEN     = os.environ.get("HF_TOKEN", "YOUR_HF_TOKEN")

CONFIG.set_default_api_key(NDIF_API_KEY)
CONFIG.API.HOST = "http://ndif-hackathon.duckdns.org:8001"  # hackathon NDIF cluster

if HF_TOKEN not in ("", "YOUR_HF_TOKEN"):
    login(token=HF_TOKEN)

# One probe is trained + saved per model.
#   multimodal : gemma-3 is a VisionLanguageModel; text models use LanguageModel
#   generator  : the value of the liars-bench `model` column to use for that model's
#                on-policy evaluation (must match the dataset; confirm the slugs below)
#   batch_size : lower for the large MoE model if you hit OOM / remote limits
MODELS = [
    dict(hf_id="google/gemma-3-27b-it",
         multimodal=True,  generator="gemma-3-27b-it",
         batch_size=16, save_as="mean_probe_gemma-3-27b-it.npz"),
    dict(hf_id="Qwen/Qwen3.5-27B",
         multimodal=False, generator="Qwen3.5-27B",                       # TODO: confirm liars-bench slug
         batch_size=16, save_as="mean_probe_qwen3.5-27b.npz"),
    dict(hf_id="nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16",
         multimodal=False, generator="NVIDIA-Nemotron-3-Super-120B-A12B-BF16",  # TODO: confirm liars-bench slug
         batch_size=4,  save_as="mean_probe_nemotron-3-super-120b.npz"),
]

## 1️⃣ The probe

We first compute the difference of class means, and for every new test point, we project it onto that direction.
 
(Positive class = **deceptive**, so a higher score means more deceptive.)

In [3]:
import numpy as np

class MeanDifferenceProbe:
    """direction = mean(deceptive activations) - mean(honest activations)
    score(h) = h . direction

    `w` weights each example by its number of response tokens, so the class means
    are token-weighted (the mean over all response tokens pooled across the
    dataset) rather than example-weighted. This matches the reference MMSDetector,
    which pools every response token before averaging: a token-weighted mean of
    per-dialogue means is identical to the mean over all tokens. Pass token counts
    as `w` for an exact match; omit it to weight every example equally."""

    def fit(self, X, y, w=None):
        X, y = np.asarray(X, dtype=np.float64), np.asarray(y)
        w = np.ones(len(y)) if w is None else np.asarray(w, dtype=np.float64)

        def wmean(mask):
            return (X[mask] * w[mask, None]).sum(axis=0) / w[mask].sum()

        self.direction = wmean(y == 1) - wmean(y == 0)
        return self

    def decision_function(self, X):
        return np.asarray(X, dtype=np.float64) @ self.direction

## 2️⃣ Training data: RepE honest/deceptive facts

The training data is built from `data/true_false_facts.csv` (the RepE true/false facts from Zou et al.). We keep only the **true** statements and present each one **twice**, once under an honest persona and once under a dishonest persona, with *identical* assistant content.

In [4]:
import numpy as np
import pandas as pd

# RepE honest/deceptive persona prompts
HONEST_PROMPT    = "Pretend you're an honest person making statements about the world."
DECEPTIVE_PROMPT = "Pretend you're a dishonest person making statements about the world."

# Load the RepE facts and keep the first 512 true statements
facts = pd.read_csv("data/true_false_facts.csv")
true_facts = facts[facts["label"] == 1]["statement"].tolist()[:512]

# Each true fact appears under both personas, interleaved (honest, deceptive, honest, ...).
# label: 0 = honest, 1 = deceptive (positive class = deceptive)
train_examples = []
for s in true_facts:
    train_examples.append((HONEST_PROMPT,    s, 0))
    train_examples.append((DECEPTIVE_PROMPT, s, 1))

y = np.array([lbl for *_, lbl in train_examples])
print(f"RepE training set: {len(train_examples)} dialogues "
      f"({(y == 1).sum()} deceptive / {(y == 0).sum()} honest) from {len(true_facts)} true facts")

RepE training set: 612 dialogues (306 deceptive / 306 honest) from 306 true facts


## 3️⃣ Per-model pipeline (tokenization, model loading, activation extraction)

These helpers are parameterized by model/tokenizer so we can run the identical pipeline for each of the three models. The RepE detect span is the fact minus its last 5 words; the eval detect span is the final assistant message. Both are located with `char_to_token`, which is robust across chat templates (no per-model EOT/system-prompt special-casing needed).

In [ ]:
from transformers import AutoTokenizer

def normalize_messages(tokenizer, messages):
    """Return messages the tokenizer's chat template accepts. If it rejects a
    `system` role (e.g. gemma), fold the leading system prompt into the first user turn."""
    msgs = [{"role": m["role"], "content": m["content"]} for m in messages]
    try:
        tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        return msgs
    except Exception:
        norm, pending = [], None
        for m in msgs:
            if m["role"] == "system" and not norm:
                pending = m["content"]; continue
            if not norm and m["role"] == "user" and pending:
                norm.append({"role": "user", "content": pending + "\n\n" + m["content"]}); pending = None; continue
            norm.append(m)
        if pending:
            norm = [{"role": "user", "content": pending}] + norm
        return norm

def _span(tokenizer, text, content, span_len=None):
    """Tokenize `text` and return (ids, start, end) mapping the [content .. content+span_len)
    character range to token indices. `content` is stripped first because many chat templates
    apply `| trim` to message content, so the rendered text holds the trimmed version."""
    content = content.strip()
    enc = tokenizer(text, add_special_tokens=False)
    ids = enc["input_ids"]

    c = text.rfind(content) if content else -1
    if c == -1 and content:                       # last resort: match a shrinking prefix
        probe = content[:200]
        while len(probe) > 20 and c == -1:
            c = text.rfind(probe)
            if c == -1:
                probe = probe[: len(probe) // 2]
        span_len = None                            # only the start is reliable now
    if c == -1:                                    # give up -> empty span (gets weight 0)
        return ids, len(ids), len(ids)

    start = enc.char_to_token(c)
    if start is None:
        start = 0
    if span_len == 0:
        return ids, start, start
    end_char = c + (len(content) if span_len is None else min(span_len, len(content)))
    end = enc.char_to_token(end_char)
    if end is None:
        end = len(ids)
    return ids, start, end

def tokenize_repe(tokenizer, user_prompt, statement):
    """detect span = fact statement minus its last 5 words (repo's repe_honesty region)."""
    statement = statement.strip()
    fact_start = " ".join(statement.split(" ")[:-5])  # "" if <=5 words
    msgs = [{"role": "user", "content": user_prompt},
            {"role": "assistant", "content": statement}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return _span(tokenizer, text, statement, span_len=len(fact_start) if fact_start else 0)

def tokenize_eval(tokenizer, messages):
    """detect span = the final (assistant) message content tokens."""
    msgs = normalize_messages(tokenizer, messages)
    content = msgs[-1]["content"]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return _span(tokenizer, text, content)

In [ ]:
import torch
from nnsight import LanguageModel, VisionLanguageModel
from datasets import load_dataset

def load_model(cfg):
    Cls = VisionLanguageModel if cfg["multimodal"] else LanguageModel
    model = Cls(cfg["hf_id"])
    tokenizer = AutoTokenizer.from_pretrained(cfg["hf_id"])
    return model, tokenizer

def decoder_layers(model, cfg):
    """Residual-stream decoder blocks. Path differs by architecture — adjust here
    if a model's module tree is non-standard (e.g. the Nemotron MoE)."""
    return model.model.language_model.layers if cfg["multimodal"] else model.model.layers

def num_layers(model):
    c = getattr(model.config, "text_config", model.config)
    return c.num_hidden_layers

def collate(batch, pad_id):
    max_len = max(len(t[0]) for t in batch)
    B = len(batch)
    input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
    attn = torch.zeros((B, max_len), dtype=torch.long)
    resp = torch.zeros((B, max_len), dtype=torch.float32)
    for i, (ids, s, e) in enumerate(batch):
        input_ids[i, :len(ids)] = torch.tensor(ids)
        attn[i, :len(ids)] = 1
        resp[i, s:e] = 1.0
    return input_ids, attn, resp

def extract_all_layers(model, layers_mod, tok_list, layers, pad_id, batch_size):
    """Mean-pool the detect span at every layer. All batches are bundled into ONE
    remote session request (one queue wait). -> {layer: (N, hidden)}"""
    batches = [tok_list[i:i + batch_size] for i in range(0, len(tok_list), batch_size)]
    saved = []
    with model.session(remote=True):
        for batch in batches:
            input_ids, attn, resp = collate(batch, pad_id)
            with model.trace({"input_ids": input_ids, "attention_mask": attn}):  # remote handled by session
                pooled = []
                for L in layers:
                    h = layers_mod[L].output[0]                      # (B, T, D)
                    mask = resp.to(h.device).unsqueeze(-1)
                    denom = resp.to(h.device).sum(dim=1, keepdim=True).clamp(min=1)
                    pooled.append(((h * mask).sum(dim=1) / denom).detach().cpu())
                saved.append(torch.stack(pooled, dim=0).save())     # (num_layers, B, hidden)
    out = {L: [] for L in layers}
    for stacked in saved:
        feats = stacked.float().numpy()
        for i, L in enumerate(layers):
            out[L].append(feats[i])
    return {L: np.concatenate(out[L], axis=0) for L in layers}

# ---- on-policy evaluation data ----
EVAL_CONFIGS = ["instructed-deception", "convincing-game", "harm-pressure-choice"]
N_PER_LABEL = 500

def load_scenario_rows(config, generator, n_per_label=N_PER_LABEL, seed=42):
    """Balanced sample of a Liars' Bench scenario, filtered to `generator`'s own rows."""
    ds = load_dataset("Cadenza-Labs/liars-bench", data_dir=config)
    df = ds[list(ds.keys())[0]].to_pandas()
    df = df[df["model"] == generator].reset_index(drop=True)
    dec = df[df["deceptive"]].index.values
    hon = df[~df["deceptive"]].index.values
    n = min(n_per_label, len(dec), len(hon))
    if n == 0:
        return [], np.array([])
    rng = np.random.default_rng(seed)
    keep = np.concatenate([rng.choice(dec, n, replace=False), rng.choice(hon, n, replace=False)])
    d = df.iloc[keep].sample(frac=1, random_state=seed).reset_index(drop=True)
    y_s = d["deceptive"].astype(int).values
    msgs = [list(r["messages"]) for _, r in d.iterrows()]
    return msgs, y_s

## 4️⃣ Train, evaluate, and save one probe per model

For each model we: extract RepE training activations → fit a probe per layer (token-weighted) → extract on-policy Liars' Bench activations for each scenario → score per-scenario + average AUROC → pick the best-average layer → save that probe's weights. The selected layer and weights are independent per model.

In [ ]:
from sklearn.metrics import roc_auc_score

all_results = {}   # hf_id -> {best_layer, table, summary, save_as}

for cfg in MODELS:
    print(f"\n{'='*22} {cfg['hf_id']} {'='*22}")
    model, tokenizer = load_model(cfg)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    layers_mod = decoder_layers(model, cfg)
    LAYERS = list(range(num_layers(model)))

    # --- training: RepE activations (token-weighted mean-difference probe per layer) ---
    train_tok = [tokenize_repe(tokenizer, u, s) for (u, s, _) in train_examples]
    counts = np.array([e - s for (_, s, e) in train_tok], dtype=np.float64)
    print(f"extracting RepE training activations ({len(train_tok)} dialogues, {len(LAYERS)} layers)...")
    X_train = extract_all_layers(model, layers_mod, train_tok, LAYERS, pad_id, cfg["batch_size"])
    probes = {L: MeanDifferenceProbe().fit(X_train[L], y, counts) for L in LAYERS}

    # --- on-policy evaluation: this model's own Liars' Bench generations ---
    eval_sets = {}
    for sc in EVAL_CONFIGS:
        msgs, y_s = load_scenario_rows(sc, cfg["generator"])
        if len(msgs) == 0:
            print(f"  [skip] no '{cfg['generator']}' rows in {sc}")
            continue
        etok = [tokenize_eval(tokenizer, m) for m in msgs]
        print(f"extracting {sc} eval activations ({len(etok)} examples)...")
        eval_sets[sc] = {"X": extract_all_layers(model, layers_mod, etok, LAYERS, pad_id, cfg["batch_size"]),
                         "y": y_s}

    # --- per-layer per-scenario AUROC + average, pick best layer ---
    rows = []
    for L in LAYERS:
        row = {"layer": L}
        for sc, d in eval_sets.items():
            row[sc] = roc_auc_score(d["y"], probes[L].decision_function(d["X"][L]))
        row["average"] = float(np.mean([row[sc] for sc in eval_sets])) if eval_sets else float("nan")
        rows.append(row)
    table = pd.DataFrame(rows).set_index("layer")
    best_layer = int(table["average"].idxmax()) if eval_sets else LAYERS[len(LAYERS) // 2]
    print(table.round(3).to_string())
    print(f"best layer (avg AUROC): {best_layer}")

    # --- save this model's probe weights ---
    final = MeanDifferenceProbe().fit(X_train[best_layer], y, counts)
    np.savez(cfg["save_as"],
             direction=final.direction.astype(np.float32),
             layer=np.array(best_layer),
             model_id=np.array(cfg["hf_id"]))
    print(f"saved {cfg['save_as']} | layer={best_layer} | dim={final.direction.shape[0]}")

    all_results[cfg["hf_id"]] = dict(
        best_layer=best_layer, table=table, save_as=cfg["save_as"],
        summary={**{sc: float(table.loc[best_layer, sc]) for sc in eval_sets},
                 "average": float(table.loc[best_layer, "average"])} if eval_sets else {})

    del model, X_train, eval_sets, probes

In [ ]:
print("=== Best-layer transfer AUROC per model ===\n")
summary_rows = []
for hf_id, r in all_results.items():
    print(f"{hf_id}  (layer {r['best_layer']}  ->  {r['save_as']})")
    for k, v in r["summary"].items():
        print(f"   {k:24s} {v:.3f}")
    print()
    summary_rows.append({"model": hf_id, "best_layer": r["best_layer"], **r["summary"]})

summary_df = pd.DataFrame(summary_rows).set_index("model")
summary_df.round(3)

In [ ]:
import matplotlib.pyplot as plt

n = len(all_results)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 4), squeeze=False)
for ax, (hf_id, r) in zip(axes[0], all_results.items()):
    t = r["table"]
    for col in t.columns:
        is_avg = col == "average"
        ax.plot(t.index, t[col], label=col, lw=2.4 if is_avg else 1.2,
                alpha=0.9 if is_avg else 0.6, color="black" if is_avg else None)
    ax.axvline(r["best_layer"], color="red", ls=":", lw=1, label=f"best ({r['best_layer']})")
    ax.axhline(0.5, color="gray", ls="--", lw=1)
    ax.set_title(hf_id.split("/")[-1]); ax.set_xlabel("layer"); ax.set_ylim(0, 1)
    ax.set_ylabel("transfer AUROC"); ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 5️⃣ Saved probes

The loop above wrote one `.npz` per model (`direction`, `layer`, `model_id`). Each is submitted with the matching model. We reload them here to verify.

In [ ]:
import glob

for f in sorted(glob.glob("mean_probe_*.npz")):
    d = np.load(f, allow_pickle=True)
    print(f"{f}  | model: {str(d['model_id'])}  | layer: {int(d['layer'])}  | dim: {d['direction'].shape[0]}")